# Statistics Villains: Web Scraping with pandas
**Big idea:** extract a table from a webpage, then explain what its rows mean.

This standalone lesson uses the instructor's [Statistics Villains website](https://people.clas.ufl.edu/mathguy/supervillain-statistics-playground/). Learn to recognize HTML, request a page responsibly, clean a table, and make one graph. The default is a six-row teaching excerpt checked on October 24, 2026. It contains fictional villains and invented costs, not real insurance data.

**Understand → Predict → Run → Change → Use → Practice → Connect**

[View on GitHub](https://github.com/mathguy-uf/MAD2502/blob/main/applied_data/03_statistics_villains.ipynb) · [Run in Google Colab](https://colab.research.google.com/github/mathguy-uf/MAD2502/blob/main/applied_data/03_statistics_villains.ipynb) · [Launch Binder/JupyterLab](https://mybinder.org/v2/gh/mathguy-uf/MAD2502/main?urlpath=lab/tree/applied_data/03_statistics_villains.ipynb)

Run cells from the top with **Shift+Enter**. Restart the kernel and run all to check your work. Each tutorial includes its own imports and data. Hosted sessions are temporary: download your work before leaving.

In [ ]:
# If an import fails, uncomment and run this once, then restart the kernel.
# %pip install pandas matplotlib requests lxml

## Understand — website, HTML, and tables
A **website** is a collection of webpages. **HTML** (HyperText Markup Language) describes their structure. A browser displays that structure; `requests` downloads the server's response without running a browser's JavaScript.

A **tag** marks an element: `<table>` contains a table, `<tr>` a row, `<th>` a heading, and `<td>` a data cell. `response.text` is HTML text. An API or downloadable CSV is usually more stable and easier to interpret than presentation-oriented HTML.

`robots.txt` gives automated-access instructions for a site. Read those and the site's terms before a live request. It is not a data license. The optional code checks the exact page with the same user agent sent in the request. A failed policy check stops the live attempt.

A pandas **DataFrame** is a table with named columns. A Python **list** holds ordered items, indexed from 0. The setup imports packages and embeds a tiny version of the instructor's table, so you can learn the same workflow without network access. `RUN_LIVE = False` selects this sample. Your conclusions must say which source was used.

In [ ]:
from io import StringIO
from urllib.robotparser import RobotFileParser
import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display

PAGE_URL = "https://people.clas.ufl.edu/mathguy/supervillain-statistics-playground/"
ROBOTS_URL = "https://people.clas.ufl.edu/robots.txt"
USER_AGENT = "MAD2502Teaching/1.0"
RUN_LIVE = False
SAMPLE_HTML = '<table><thead><tr><th>Villain Name</th><th>Insurance Cost</th><th>Number of Henchmen</th><th>Villain Category</th></tr></thead><tbody>\n<tr><td>Condiment King</td><td>$7,900</td><td>2</td><td>Human</td></tr>\n<tr><td>Calendar Man</td><td>$2,500</td><td>3</td><td>Human</td></tr>\n<tr><td>Polka-Dot Man</td><td></td><td></td><td>Human</td></tr>\n<tr><td>Kite Man</td><td></td><td></td><td>Human</td></tr>\n<tr><td>Crazy Quilt</td><td>$9,600</td><td>1</td><td>Human</td></tr>\n<tr><td>Orca</td><td>$9,100</td><td>1</td><td>Mutate</td></tr>\n</tbody></table>'
html = SAMPLE_HTML
source_label = "six-row teaching excerpt; not the complete page"

## Predict — what comes back?
Will downloading a webpage give a DataFrame immediately? If HTML contains three tables, how many tables might pandas return?

HTTP 200 usually means success; 403 means forbidden; 404 means not found; 429 means too many requests. `raise_for_status()` turns HTTP error statuses into exceptions. The timeout limits waiting. Change `RUN_LIVE` above only after checking the website's access terms.

In [ ]:
if RUN_LIVE:
    try:
        robots = requests.get(ROBOTS_URL, headers={"User-Agent": USER_AGENT}, timeout=30)
        robots.raise_for_status()
        policy = RobotFileParser()
        policy.parse(robots.text.splitlines())
        if not policy.can_fetch(USER_AGENT, PAGE_URL):
            raise ValueError("robots.txt does not allow this request.")
        response = requests.get(PAGE_URL, headers={"User-Agent": USER_AGENT}, timeout=30)
        response.raise_for_status()
        html = response.text
        source_label = "live instructor webpage"
    except (requests.RequestException, ValueError) as error:
        print("Live retrieval unavailable:", error)
        print("Using the labeled teaching excerpt instead.")
print("Source:", source_label)
print("HTML type:", type(html))
print(html[:200])

## Run — turn HTML into tables
`StringIO` wraps text so pandas can read it like a file. `pd.read_html(...)` returns a **list of DataFrames**, since a page can have several tables. `enumerate` gives each table's position and value. We inspect the headings and choose the table with the expected fields, rather than blindly assuming the first one is correct.

The table-selection code is reusable setup. A set holds the required column names; `issubset` checks that all are present. If the site changes, stop and inspect rather than analyzing a different table by accident.

In [ ]:
tables = pd.read_html(StringIO(html))
for index, table in enumerate(tables):
    print(index, table.shape, list(table.columns))
required_columns = {"Villain Name", "Insurance Cost", "Number of Henchmen"}
matches = [table for table in tables if required_columns.issubset(table.columns)]
if len(matches) != 1:
    raise ValueError("The page structure changed: inspect tables before continuing.")
villains = matches[0].copy()
display(villains.head())

## Change — clean one thing at a time
One row represents one fictional villain. `.copy()` keeps our cleaning separate from the original table. Column names are labels; we remove outer spaces, lowercase them, and replace spaces with underscores. **Predict:** what does `Insurance Cost` become?

Money arrives as text because of `$` and commas. `.str.replace(...)` removes those literal characters. `pd.to_numeric(..., errors="coerce")` converts valid numbers and marks unconvertible values missing. Missing costs are unknown, not zero.

In [ ]:
villains.columns = [str(name).strip().lower().replace(" ", "_") for name in villains.columns]
assert villains.columns.is_unique, "Cleaning created repeated column names."
cost_text = villains["insurance_cost"].astype("string")
cost_text = cost_text.str.replace("$", "", regex=False).str.replace(",", "", regex=False)
villains["insurance_cost"] = pd.to_numeric(cost_text, errors="coerce")
villains["number_of_henchmen"] = pd.to_numeric(villains["number_of_henchmen"], errors="coerce")
print(villains.dtypes)
print("Missing values:")
print(villains.isna().sum())
display(villains)

## Use — ask, select, compute, check, communicate
Which villains have a known insurance cost of at least $8,000? The Boolean comparison selects rows. `sort_values` orders them; `mean` and `median` summarize known costs. State the number of known values alongside any summary.

In [ ]:
expensive = villains[villains["insurance_cost"] >= 8000]
display(expensive.sort_values("insurance_cost", ascending=False))
print("Known costs:", villains["insurance_cost"].count(), "of", len(villains))
print("Mean:", villains["insurance_cost"].mean())
print("Median:", villains["insurance_cost"].median())

### Plot known costs
`dropna(subset=[...])` removes only rows missing the variable needed for this graph. It does not change `villains`. A horizontal bar chart has room for names. Labels and the source note make it interpretable without color. **Predict:** how many bars will the excerpt produce?

In [ ]:
plot_data = villains.dropna(subset=["insurance_cost"]).sort_values("insurance_cost").tail(10)
ax = plot_data.plot.barh(x="villain_name", y="insurance_cost", legend=False, figsize=(8, 4))
ax.set(xlabel="Fictional insurance cost ($)", ylabel="Villain", title="Known costs — " + source_label)
plt.tight_layout()
plt.show()

## Practice
1. Change the threshold from 8,000 to 9,000. Predict which rows remain.
2. Count the missing henchman values. Why would filling them with zero change the meaning?
3. Change the plot to known henchman counts and update its labels.
4. Explain why an excerpt mean is not necessarily the full-page mean.

## Connect
Once HTML becomes a DataFrame, the same workflow applies to a CSV or API. Record the URL, access date, selected table, row definition, and cleaning decisions in your project journal. A page redesign may require revisiting the table-selection step.

[Current pandas read_html reference](https://pandas.pydata.org/docs/reference/api/pandas.read_html.html)